In [5]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

# 1.加载环境变量
load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 3.定义tool
# 描述方式一：使用tool装饰器的属性描述
@tool("square_root", description="求x的平方根")
def square_root(x: float) -> float:
    return x ** 0.5


# 描述方式二：使用文档注释描述
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    获取当前天气和可选的天气预报。
    Args:
        location: 城市名称或坐标
        units: 温度单位，可选值['celsius', 'fahrenheit']
        include_forecast: 包括天气预报吗
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


# 描述方式三：使用BaseModel描述
from pydantic import BaseModel, Field
from typing import Literal
from langchain.tools import tool


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="城市名称或坐标")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="温度单位"
    )
    include_forecast: bool = Field(
        default=False,
        description="包括天气预报吗"
    )


# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather2(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """获取当前天气和可选的天气预报。"""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


tools = [square_root, get_weather, get_weather2]

# 4.创建智能体
agent = create_agent(model, tools)

# 5.调用模型，流式调用
stream = agent.stream_events(
    # {"messages": [HumanMessage("帮我计算429的平方根")]},
    {"messages": [HumanMessage("广州未来几天的天气如何")]},
    version="v3"
)

for message in stream.messages:
    for chunk in message.text:
        print(chunk, end="", flush=True)

I'll check the weather forecast for Guangzhou for you.广州未来几天的天气情况如下：

- **当前天气**：22°C
- **未来5天**：晴

整体来看天气晴朗稳定，气温舒适（22°C左右），适合外出活动。不过广州近期天气可能昼夜温差较大，建议出行时注意查看每日具体温度变化。

需要我查询更详细的逐日预报（比如具体温度范围、湿度等）吗？